用于信号生成、过滤和处理的高性能计算模块。

# generate_nb

使用信号选择函数 `choice_func_nb` 生成 `bool` 数组。
- `choice_func_nb` 的函数签名：func(from_i, to_i, col, *args) -> np.array
  - 参数
    - from_i：搜索范围起始索引(包含)
    - to_i：搜索范围结束索引(不包含)
    - col：当前列索引
    - *args：传递给选择函数的额外参数
  - 返回值：索引数组，范围在[from_i, to_i)内
  - 该函数根据 `from_i, to_i, col` 生成序号表示行索引，这些行索引处须为 `True`
  

参数
- `shape` (tuple)：目标信号矩阵的形状，格式为(行数, 列数)
- `pick_first` (bool)：是否只选择choice_func_nb返回的第一个信号
  - True：只取第一个信号，用于单次信号生成
  - False：取所有信号，用于多次信号生成
- `choice_func_nb` (callable)：信号选择函数，必须是Numba编译的函数
- `*args`：传递给 choice_func_nb 的额外参数

返回值：np.array，形状为 shape 的布尔数组，True 表示信号位置

## 源码

```python
@njit
def generate_nb(shape: tp.Shape,
                pick_first: bool,
                choice_func_nb: tp.ChoiceFunc, *args) -> tp.Array2d:

    out = np.full(shape, False, dtype=np.bool_)

    for col in range(out.shape[1]):
        idxs = choice_func_nb(0, shape[0], col, *args)
        if len(idxs) == 0:
            continue
        if pick_first:
            first_i = idxs[0]
            if first_i < 0 or first_i >= shape[0]:
                raise ValueError("First returned index is out of bounds")
            out[first_i, col] = True
        else:
            if np.any(idxs < 0) or np.any(idxs >= shape[0]):
                raise ValueError("Returned indices are out of bounds")
            out[idxs, col] = True
    return out
```

## 例子

In [ ]:
import numpy as np
from numba import njit
from vectorbt.signals.nb import generate_nb

@njit
def choice_func_nb(from_i, to_i, col):
    # 每列在不同位置生成信号
    return np.array([from_i + col])

# 生成3列5行的信号矩阵
signals = generate_nb((5, 3), False, choice_func_nb)
print(signals)

# generate_ex_nb

参数
- `entries` (np.array)：入场信号的布尔数组，形状为(时间, 资产)
- `wait` (int)：退出信号延迟周期数
  - 0：允许在同一周期内入场和退出
  - 大于 0：必须等待指定周期后才能退出

  注意：wait=0可能导致同一bar出现两个信号
- `until_next` (bool)：是否只在下一个入场信号前搜索退出信号
  - True：限制搜索范围到下一个入场信号
  - False：搜索到序列末尾

  注意：False时难以判断退出信号属于哪个入场信号
- `skip_until_exit` (bool)：是否跳过处理退出前的入场信号。只在until_next=False时有效
  - True：跳过退出前的新入场信号
  - False：处理所有入场信号

  注意：True时难以判断退出信号属于哪个入场信号
- `pick_first` (bool)：是否只选择退出选择函数返回的第一个信号
- `exit_choice_func_nb` (callable)：退出信号选择函数，必须是Numba编译的函数
  - 参见 generate_nb 中 choice_func_nb 的说明
- `*args`: 传递给 exit_choice_func_nb 的额外参数

返回：np.array，与 entries 相同形状的布尔数组，True 表示退出信号位置

## 源码

```python
@njit
def generate_ex_nb(entries: tp.Array2d,
                   wait: int,
                   until_next: bool,
                   skip_until_exit: bool,
                   pick_first: bool,
                   exit_choice_func_nb: tp.ChoiceFunc, *args) -> tp.Array2d:

    exits = np.full_like(entries, False)

    for col in range(entries.shape[1]):
        entry_idxs = np.flatnonzero(entries[:, col])
        last_exit_i = -1
        for i in range(entry_idxs.shape[0]):
            # Calculate the range to choose from
            if skip_until_exit and entry_idxs[i] <= last_exit_i:
                continue
            from_i = entry_idxs[i] + wait
            if i < entry_idxs.shape[0] - 1 and until_next:
                to_i = entry_idxs[i + 1]
            else:
                to_i = entries.shape[0]
            if to_i > from_i:
                # Run the UDF
                idxs = exit_choice_func_nb(from_i, to_i, col, *args)
                if len(idxs) == 0:
                    continue
                if pick_first:
                    first_i = idxs[0]
                    if first_i < from_i or first_i >= to_i:
                        raise ValueError("First returned index is out of bounds")
                    exits[first_i, col] = True
                    last_exit_i = first_i
                else:
                    if np.any(idxs < from_i) or np.any(idxs >= to_i):
                        raise ValueError("Returned indices are out of bounds")
                    exits[idxs, col] = True
                    last_exit_i = idxs[-1]
    return exits
```

## 例子

In [3]:
import numpy as np
from numba import njit
from vectorbt.signals.nb import generate_ex_nb

@njit
def exit_after_3_bars(from_i, to_i, col):
    if from_i + 2 < to_i:
        return np.array([from_i + 2], dtype=np.int64)
    return np.empty(0, dtype=np.int64)  # 推荐用 np.empty 而不是 np.array([])

# 你的 entries
entries = np.array([[True, False], [False, False], 
                    [False, False], [False, False]])
exits = generate_ex_nb(entries, 1, True, False, True, 
                        exit_after_3_bars)
print(exits)

[[False False]
 [False False]
 [False False]
 [ True False]]
